# GReaT Synthetic Data Generation (T4 GPU)

This notebook runs GReaT on Google Colab's free T4 GPU for the synthetic data study.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Upload your `train_data.csv` file when prompted
3. Run all cells
4. Download the generated synthetic data files

**Expected runtime:** ~55-65 min per replicate, ~5-6 hours total for 5 replicates

**Hyperparameters:**
- Model: distilgpt2
- Epochs: 15 (GReaT converges quickly on tabular data)
- Batch size: 64

In [ ]:
# Install dependencies
!pip install -q be-great pandas numpy torch

In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Upload training data
from google.colab import files

print("Please upload your train_data.csv file:")
uploaded = files.upload()

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
from be_great import GReaT

warnings.filterwarnings('ignore')

# Load training data
df_train = pd.read_csv('train_data.csv')
print(f"Loaded training data: {len(df_train):,} records, {len(df_train.columns)} columns")
print(f"Columns: {list(df_train.columns)}")
df_train.head()

In [ ]:
# Configuration
RANDOM_SEEDS = [42, 123, 456, 789, 1011]
N_REPLICATES = 5
N_SAMPLES = len(df_train)  # Generate same number as training data

# GReaT hyperparameters (optimized for T4 GPU runtime)
# Note: GReaT converges quickly on tabular data; 15 epochs is sufficient
LLM = 'distilgpt2'
EPOCHS = 15  # Reduced for feasibility (~1 hour per replicate)
BATCH_SIZE = 64  # Increased for faster training

# Calculate expected steps
steps_per_epoch = len(df_train) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS

print("Configuration:")
print(f"  Replicates: {N_REPLICATES}")
print(f"  Samples per replicate: {N_SAMPLES:,}")
print(f"  LLM: {LLM}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Steps per epoch: {steps_per_epoch:,}")
print(f"  Total steps: {total_steps:,}")
print(f"  Estimated time: ~{total_steps / 2.64 / 60:.0f} min per replicate")

In [ ]:
def set_all_seeds(seed):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def generate_great_replicate(df_train, n_samples, seed, rep_num):
    """Generate one replicate of GReaT synthetic data."""
    print(f"\n{'='*60}")
    print(f"Replicate {rep_num}/5 (seed={seed})")
    print(f"{'='*60}")

    # Set seeds
    set_all_seeds(seed)

    # Create and train model
    print("Training GReaT model...")
    model = GReaT(
        llm=LLM,
        experiment_dir=f'great_rep{rep_num}',
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        seed=seed,
        data_seed=seed,
    )
    model.fit(df_train)

    # Generate synthetic data
    print(f"Generating {n_samples:,} synthetic records...")
    set_all_seeds(seed + 1000)  # Different seed for generation
    df_synth = model.sample(
        n_samples=n_samples,
        temperature=0.7,
        k=100,
        device='cuda',
        drop_nan=True,
    )

    # Ensure column order matches training data
    available_cols = [c for c in df_train.columns if c in df_synth.columns]
    df_synth = df_synth[available_cols]

    print(f"Generated {len(df_synth):,} records")
    return df_synth

In [ ]:
# Run all replicates
import time

synthetic_datasets = {}
total_start = time.time()

for rep in range(N_REPLICATES):
    seed = RANDOM_SEEDS[rep]
    rep_num = rep + 1

    start_time = time.time()
    df_synth = generate_great_replicate(df_train, N_SAMPLES, seed, rep_num)
    elapsed = time.time() - start_time

    # Save to dict
    synthetic_datasets[f'great_rep{rep_num}'] = df_synth

    # Save to CSV
    filename = f'great_rep{rep_num}.csv'
    df_synth.to_csv(filename, index=False)
    print(f"Saved to {filename} ({elapsed/60:.1f} min)")

total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"COMPLETE! Total time: {total_elapsed/60:.1f} min ({total_elapsed/3600:.2f} hours)")
print(f"{'='*60}")

In [ ]:
# Quick quality check
print("Sample synthetic data from replicate 1:")
print(synthetic_datasets['great_rep1'].head(10))
print("\nColumn dtypes:")
print(synthetic_datasets['great_rep1'].dtypes)

In [ ]:
# Download all synthetic datasets
from google.colab import files

print("Downloading synthetic data files...")
for rep in range(1, N_REPLICATES + 1):
    filename = f'great_rep{rep}.csv'
    files.download(filename)
    print(f"Downloaded {filename}")

print("\nDone! Copy these files to: results/experiments/synthetic_data/")

## Next Steps

1. Download all `great_rep*.csv` files
2. Copy them to your local project: `results/experiments/synthetic_data/`
3. Run measures locally:
   ```bash
   tsd run --methods great
   ```